# Phase 03 — Literature & Baseline Research
## CodeMentor-LLM
Testing **meta-llama/Llama-3.2-3B-Instruct** base model on 10 coding questions
before any fine-tuning to justify my project.

#  Libraries

In [ ]:
!pip install -q transformers==4.49.0 bitsandbytes==0.45.3 accelerate==1.5.1 peft==0.14.0

# Checking GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

# Login to HuggingFace

In [ ]:
from huggingface_hub import login
login()

# Load meta-llama/Llama-3.2-3B-Instruct in 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# 4-bit quantization config --> QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Load in 4 bits
    bnb_4bit_quant_type="nf4",             # Special format called NormalFloat4 --> better accuracy than normal 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16, # During calculation, use FP16 for stability
    bnb_4bit_use_double_quant=True,        # Compresses even the quantization values
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto", # Split model layers smartly --> GPU and CPU
)

print("Model loaded successfully")
print(f"Memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

# Create inference function

In [ ]:
def generate_response(
    prompt,            # User Question
    max_new_tokens=256 # maximum words/tokens model will generate
    ):

    # Apply Mistral chat template --> This converts your input into chat format.
    messages = [
        {"role": "user",
         "content": prompt}
        ]

    # Tokenize
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only new tokens
    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )
    return response

print("Inference function ready")

# Run 10 coding questions against base model

In [ ]:
# 10 coding questions to test base model
questions = [
    "Write a Python function to reverse a string.",
    "Explain what a decorator is in Python with an example.",
    "What is the difference between a list and a tuple in Python?",
    "Write a Python function to check if a number is prime.",
    "How do you handle exceptions in Python? Show an example.",
    "Write a SQL query to find the second highest salary from a table.",
    "Explain the concept of recursion with a Python example.",
    "What is the difference between == and is in Python?",
    "Write a Python function to find duplicates in a list.",
    "Explain what a REST API is in simple terms.",
]

# Run all 10 questions
results = []
for i, question in enumerate(questions):
    print(f"\n{'='*30}")
    print(f"Q{i+1}: {question}")
    print(f"{'='*30}")
    response = generate_response(question)
    print(f"A: {response}")
    results.append({"question": question, "response": response})

print("\nAll 10 questions completed")

## Base Model Analysis — meta-llama/Llama-3.2-3B-Instruct

### What the base model does WELL:
- Answers all 10 coding questions correctly
- Formats code blocks consistently with backticks
- Includes docstrings in code (Q1, Q4, Q9) — good practice
- Uses bold headers and structured sections (Q2, Q4, Q6)
- Provides real-world analogies (Q10 — restaurant analogy)
- SQL answer uses DENSE_RANK() — more accurate than 8B version
- Correct identity operator examples (Q8)
- Handles Python, SQL, and conceptual questions well

### What the base model FAILS at:
- Responses cut off mid-sentence in 5/10 questions (Q2, Q3, Q4, Q7, Q8)
- No consistent response length
- Attention mask warning on every inference
- No specific coding instruction format
- Q10 wrong full form — "Representational State of Resource" instead of "Representational State Transfer"
- Over-explains simple concepts (Q9)
- Inconsistent output quality compared to larger models

### Why Fine-Tuning is justified:
1. Responses cut off in 50% of questions
2. Factual errors present (Q10 wrong full form)
3. No consistent coding instruction format
4. SFT on CodeAlpaca-20K will teach complete structured responses
5. DPO will align model to prefer accurate higher quality responses
6. Fine-tuned model will be domain-specific and focused

# Save results to JSON

In [ ]:
import json

# Save baseline results
baseline_results = {
    "model": "meta-llama/Llama-3.2-3B-Instruct",
    "type": "base_model_no_finetuning",
    "results": results,
    "observations": {
        "strengths": [
            "Answers all 10 questions correctly",
            "Formats code blocks with backticks",
            "Includes docstrings in code",
            "Uses bold headers and structured sections",
            "SQL uses DENSE_RANK — more accurate",
            "Handles Python, SQL, concepts well"
        ],
        "weaknesses": [
            "Responses cut off in 5/10 questions",
            "No consistent response length",
            "Attention mask warning on every inference",
            "No coding instruction format",
            "Factual error in Q10 — wrong REST full form",
            "Over-explains simple concepts"
        ],
        "finetuning_justified": True
    }
}

# Save to file
with open("baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=4)

print("Baseline results saved to baseline_results.json")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')